In [1]:
import pandas as pd

# Step 1
cube_df = pd.read_csv("../data/ab_cube_base.csv")

In [4]:
# TODO:
# 현재 cube_df는 metric 컬럼에 impression / click / purchase / revenue가
# 세로(long-format) 형태로 저장된 구조입니다.
# KPI를 계산하려면 각 metric이 하나의 컬럼으로 펼쳐진 wide-format 구조가 필요합니다.
# 따라서 pivot_table()을 사용하여
# - index: variant, country, device
# - columns: metric
# - values: sum_value
# 로 변환하세요.
pivot_df = cube_df.pivot_table(
    index = ['variant', 'country', 'device'],
    columns='metric',
    values='sum_value'
)

In [5]:
# TODO:
# pivot 결과는 index 기반 구조이므로,
# 이후 merge와 컬럼 계산을 쉽게 하기 위해 reset_index()를 적용하세요.
pivot_df = pivot_df.reset_index()

In [6]:
# TODO:
# pivot 결과 컬럼명을 아래 순서로 명확하게 지정하세요.
# ["variant", "country", "device", "click", "impression", "purchase", "revenue"]
# 이렇게 해야 이후 KPI 계산 시 어떤 컬럼이 무엇을 의미하는지 분명해집니다.
pivot_df.columns = ["variant", "country", "device", "click", "impression", "purchase", "revenue"]

In [7]:
# TODO:
# ARPU는 revenue를 user 수로 나누어 계산해야 하므로 count_n 정보가 필요합니다.
# count_n은 metric마다 같은 값이 반복되므로,
# variant, country, device, count_n만 선택한 뒤 중복을 제거하여 count_df를 만드세요.
# 즉, "조합별 사용자 수 테이블"을 따로 추출하는 단계입니다.
count_df = cube_df[['variant', 'country', 'device', 'count_n']].drop_duplicates()

In [8]:
# TODO:
# pivot_df와 count_df를 variant, country, device 기준으로 병합하세요.
# 이렇게 하면 KPI 계산에 필요한 click, impression, purchase, revenue, count_n이
# 모두 하나의 테이블에 모이게 됩니다.
pivot_df = pd.merge(
    pivot_df,
    count_df,
    on=['variant','country','device'],
    how='inner'
)

In [9]:
# TODO:
# CTR을 계산하세요.
# CTR은 click / impression 입니다.
# 즉, 세그먼트별 전체 클릭 수를 전체 노출 수로 나눈 값입니다.
pivot_df["CTR"] = pivot_df['click'] / pivot_df['impression']

In [10]:
# TODO:
# Conversion을 계산하세요.
# Conversion은 purchase / impression 입니다.
# 즉, 세그먼트별 전체 구매 수를 전체 노출 수로 나눈 값입니다.
pivot_df["Conversion"] = pivot_df['purchase'] / pivot_df['impression']

In [11]:
# TODO:
# ARPU를 계산하세요.
# ARPU는 revenue / count_n 입니다.
# 여기서 count_n은 해당 variant-country-device 조합의 사용자 수입니다.
# 즉, 사용자 1명당 평균적으로 얼마의 매출이 발생했는지를 계산하는 단계입니다.
pivot_df["ARPU"] = pivot_df['revenue'] / pivot_df['count_n']

In [12]:
# TODO:
# Tableau에서 KPI 선택형 시각화를 만들기 쉽도록
# CTR, Conversion, ARPU를 다시 long-format으로 변환하세요.
# - id_vars: variant, country, device
# - value_vars: CTR, Conversion, ARPU
# - var_name: metric
# - value_name: value
# 형태로 melt()를 적용하세요.
long_df = pivot_df.melt(
    id_vars=['variant', 'country', 'device'],
    value_vars=['CTR', 'Conversion', 'ARPU'],
    var_name='metric',
    value_name='value'
)

print(long_df.head())
print("shape:", long_df.shape)
print(long_df["metric"].value_counts())

   variant country   device metric     value
0        0      DE  desktop    CTR  0.152000
1        0      DE   mobile    CTR  0.207547
2        0      DE   tablet    CTR  0.212121
3        0      ES  desktop    CTR  0.187500
4        0      ES   mobile    CTR  0.196203
shape: (72, 5)
metric
CTR           24
Conversion    24
ARPU          24
Name: count, dtype: int64


In [13]:
# TODO:
# 최종 KPI long-format 데이터를 CSV 파일로 저장하세요.
# 다음 실습이나 Tableau 대시보드에서 사용할 수 있도록
# ab_kpi_long.csv 파일로 저장합니다.
long_df.to_csv("ab_kpi_long.csv", index=False)